In [1]:
# imposto il seed di random per le analisi
import random
random.seed(69)

# mappo i tipi di aggiunta in human readable
# A for added, M for modified, D for deleted
CHANGE_TYPE_MAP = {
    'A': 'Aggiunto',
    'M': 'Modificato',
    'D': 'Eliminato',
    'R': 'Rinominato',
}

# Esplorazione metriche


### carico tutti i file yaml e li associo al nome della repo

In [2]:
REPO_FOLDER = "C:\\dev\\SE4AI-base\\repositories"

import os

repo_names = sorted(
    [name for name in os.listdir(REPO_FOLDER) if os.path.isdir(os.path.join(REPO_FOLDER, name))]
)

print(sorted([f'{x.split("__")[1]}' for x in repo_names]))
print(len(repo_names))

['AIlice', 'Adala', 'Agent4Rec', 'AgentForge', 'AgentGPT', 'AgentPilot', 'AgentVerse', 'Auto-GPT', 'AutoPR', 'BabyCommandAGI', 'BambooAI', 'ChatDev', 'DemoGPT', 'DevOpsGPT', 'Devon', 'Flowise', 'GPTDiscord', 'GPTSwarm', 'GeniA', 'Godmode-GPT', 'JARVIS', 'LLMStack', 'Magick', 'MemGPT', 'MetaGPT', 'Multi-GPT', 'NLSOM', 'OpenAGI', 'OpenAgents', 'OpenDevin', 'PromethAI-Backend', 'SWE-agent', 'Suspicion-Agent', 'Teenage-AGI', 'UFO', 'Voyager', 'WrenAI', 'XAgent', 'agents', 'ai-legion', 'aider', 'autogen', 'automata', 'autonomous-hr-chatbot', 'babyagi', 'beebot', 'blinky', 'bloop', 'bondai', 'browser-extension', 'bumpgen', 'cal.com', 'camel', 'chatarena', 'chemcrow-public', 'clippy', 'codefuse-chatbot', 'cody', 'continue', 'crewai', 'data-to-paper', 'databerry', 'dev-gpt', 'developer', 'devika', 'dotagent', 'eidolon', 'english-compiler', 'evo.ninja', 'fastagency', 'friday', 'gpt-engineer', 'gpt-migrate', 'gpt-pilot', 'gpt-researcher', 'gpt-runner', 'ix', 'l2mac', 'langroid', 'lemon-agent', '

### GigaWork

gigawork nel suo output.csv ha le seguenti colonne:
* `repository`
* `commit_hash`
* `author_name`
* `author_email`
* `committer_name`
* `committer_email`
* `committed_date`
* `authored_date`
* `file_path`
* `previous_file_path`
* `file_hash`
* `previous_file_hash`
* `git_change_type`
* `valid_yaml`
* `probably_workflow`
* `valid_workflow`

una volta eseguito il run proviamo ad esplorare il dataset che ha cacciato fuori

In [3]:
# controllo quali workflow non sono vuoti e pulisco un po' la variabile repo_names
GIGAWORK_OUTPUT_DIRECTORY = "C:\\dev\\SE4AI-base\\gigawork"

empty_workflows = []

for repo in repo_names:
    repo_dir = os.path.join(GIGAWORK_OUTPUT_DIRECTORY, 'all_workflows' ,repo)
    if os.path.isdir(repo_dir):
        files = os.listdir(repo_dir)
        if not files:
            empty_workflows.append(repo)
            repo_names.remove(repo)

print(f"In queste repositories non sono stati trovati workflow: {empty_workflows}")
print(f"Numero di repository attive: {len(repo_names)}")


In queste repositories non sono stati trovati workflow: ['0xpayne__gpt-migrate', 'AutoPackAI__beebot', 'CR-Gjx__Suspicion-Agent', 'DataBassGit__AgentForge', 'LehengTHU__Agent4Rec', 'OpenBMB__XAgent', 'TheoLvs__westworldjs', 'aiwaves-cn__agents', 'amirrezasalimi__friday', 'codefuse-ai__codefuse-chatbot', 'composable-models__llm_multiagent_debate', 'ennucore__clippy', 'eylonmiz__react-agent', 'felixbrock__lemon-agent', 'gmpetrov__databerry', 'jbexta__AgentPilot', 'mczhuge__NLSOM', 'microsoft__JARVIS', 'mpaepper__llm_agents', 'pgalko__BambooAI', 'seahyinghang8__blinky', 'smol-ai__developer', 'stepanogil__autonomous-hr-chatbot', 'uilicious__english-compiler', 'yeagerai__yeagerai-agent']
Numero di repository attive: 78


In [4]:
# carico il dataset creato da gigawork
import pandas as pd

dataset_path = os.path.join(GIGAWORK_OUTPUT_DIRECTORY, "dataset.csv")
df = pd.read_csv(dataset_path)

print(df.head())

# per sicurezza controllo se ci sono repository nel dataset che non sono in repo_names (dovrebbero essere tutte in repo_names)
dataset_repos = set(df['repository'].unique())
if not dataset_repos.issubset(set(repo_names)):
    print('c\è stato un errore nell\'eliminazione delle repo')
    
# ordino repo_names in ordine alfabetico per evitare che random si comporti in modo strano
repo_names = sorted(repo_names)

                 repository                               commit_hash  \
0  AntonOsika__gpt-engineer  3bb4a49abb8aa172cb286789cd95c9b2c28160fe   
1  AntonOsika__gpt-engineer  5f9d7acc6c5a6026ea3052a3b4fc713ab597d9d2   
2  AntonOsika__gpt-engineer  df593e1b02efbfe50c35fb67f229bdb36997ccb4   
3  AntonOsika__gpt-engineer  c7827efbec9e1e22c7e02f358b8fe08875550145   
4  AntonOsika__gpt-engineer  6bdd1d176468f10c645a3caf2f9090fa76852698   

      author_name                                  author_email  \
0        captivus      366332+captivus@users.noreply.github.com   
1       ATheorell  143704446+ATheorell@users.noreply.github.com   
2       ATheorell  143704446+ATheorell@users.noreply.github.com   
3  Erik Bjäreholt                               erik@bjareho.lt   
4       ATheorell  143704446+ATheorell@users.noreply.github.com   

   committer_name           committer_email  committed_date  authored_date  \
0          GitHub        noreply@github.com      1720378754     1720378754   
1 

In [5]:
# come prima analisi controllo quanti workflow UNIVOCI ci sono per repository
# baso l'univocità sulla colonna file_path

workflow_counts = df.groupby('repository')['file_path'].nunique()
print(workflow_counts, end='\n\n')

# media per repository
average_workflows = workflow_counts.mean()
print(f"Numero medio di workflow univoci per repository: {average_workflows:.2f}")

# la repository con più workflow univoci
repo_piu_workflows = workflow_counts.sort_values(ascending=False).head(1)
print("Repository con più workflow univoci:", repo_piu_workflows.index[0], "con", repo_piu_workflows.values[0], "workflow univoci")

repository
AntonOsika__gpt-engineer         8
BloopAI__bloop                  12
Canner__WrenAI                  18
FOLLGAD__Godmode-GPT            16
Farama-Foundation__chatarena     7
                                ..
trypromptly__LLMStack            3
ur-whitelab__chemcrow-public     2
vanna-ai__vanna                  3
xeol-io__bumpgen                14
xlang-ai__OpenAgents             1
Name: file_path, Length: 71, dtype: int64

Numero medio di workflow univoci per repository: 10.51
Repository con più workflow univoci: calcom__cal.com con 96 workflow univoci


In [6]:
# prendo una repository a caso e la analizzo
# random_repo = random.choice(repo_names)
# prendo una repo rappresentativa
random_repo = "joaomdmoura__crewai"

print(f"Repository analizzata: {random_repo.split('__')[1]}")

# quali sono i workflow univoci di questa repository? e quali operazioni sono stati fatti su di esso e in quale numero?§
random_repo_workflows = df[df["repository"] == random_repo].copy()

file_counts = (
    random_repo_workflows.groupby("file_path")
    .size()
    .rename("nr di volte")
)

repeated_files = file_counts[file_counts > 1].index

if len(repeated_files) == 0:
    print("Nessun file_path ripetuto in questa repository.")
else:
    pivot_changes = (
        random_repo_workflows[random_repo_workflows["file_path"].isin(repeated_files)]
        .pivot_table(
            index="file_path",
            columns="git_change_type",
            aggfunc="size",
            fill_value=0
        )
        .reset_index()
    )

    pivot_changes = pivot_changes.rename(
        columns={c: CHANGE_TYPE_MAP.get(c, c) for c in pivot_changes.columns}
    )

    pivot_changes["nr di volte"] = pivot_changes["file_path"].map(file_counts)
    pivot_changes = pivot_changes.sort_values("nr di volte", ascending=False)

    print("Breakdown modifiche per file_path ripetuti (pivot):")
    print(pivot_changes.to_string(index=False))


Repository analizzata: crewai
Breakdown modifiche per file_path ripetuti (pivot):
                                  file_path  Aggiunto  Modificato  Rinominato  nr di volte
                .github/workflows/tests.yml         1          26           0           27
         .github/workflows/type-checker.yml         1          14           0           15
               .github/workflows/linter.yml         0          11           1           12
               .github/workflows/mkdocs.yml         0           6           1            7
              .github/workflows/publish.yml         1           5           0            6
       .github/workflows/build-uv-cache.yml         1           4           0            5
     .github/workflows/security-checker.yml         1           3           0            4
               .github/workflows/codeql.yml         1           3           0            4
                .github/workflows/stale.yml         1           3           0            4
.github/

In [7]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent  # da /notebooks -> /SE4AI-base
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
from src.util.file_name_chain import add_workflow_global_id_from_csv


In [9]:
# ma ha senso usare file_path come "chiave primaria" per identificare un workflow nel tempo? potrebbe essere che i workflow cambiano nome o vengono rinominati o cancellati spesso, controllo.

# TODO: CONTROLLARE SE CI SONO PARECCHI PREVIOUS_FILE_PATH DIVERSI DA FILE_PATH, se sono un nr significativo allora fare un algo per risalire alla catena anche quando cambiano nome magari usando i filehash e prev file hash, poi dargli un identificativo univoco a livello globale

_temp_df = add_workflow_global_id_from_csv(
    input_csv_path="C:\\dev\\SE4AI-base\\gigawork\\dataset.csv",
    output_csv_path="C:\\dev\\SE4AI-base\\gigawork\\dataset_with_ids.csv",
)

In [10]:
# conto adesso qual'è la media di workflow univoci per repository usando il nuovo dataset

workflow_counts = _temp_df.groupby('repository')['workflow_global_id'].nunique()
print(workflow_counts, end='\n\n')

# media per repository
average_workflows = workflow_counts.mean()
print(f"Numero medio di workflow univoci per repository: {average_workflows:.2f}")

# la repository con più workflow univoci
repo_piu_workflows = workflow_counts.sort_values(ascending=False).head(1)
print("Repository con più workflow univoci:", repo_piu_workflows.index[0], "con", repo_piu_workflows.values[0], "workflow univoci")

repository
AntonOsika__gpt-engineer         8
BloopAI__bloop                  13
Canner__WrenAI                  15
FOLLGAD__Godmode-GPT            13
Farama-Foundation__chatarena     7
                                ..
trypromptly__LLMStack            4
ur-whitelab__chemcrow-public     2
vanna-ai__vanna                  3
xeol-io__bumpgen                12
xlang-ai__OpenAgents             1
Name: workflow_global_id, Length: 71, dtype: int64

Numero medio di workflow univoci per repository: 9.45
Repository con più workflow univoci: calcom__cal.com con 92 workflow univoci


In [11]:
# dato che i dati adesso sono più puliti, prendo una repository a caso e la analizzo di nuovo
df = _temp_df.copy() # copio il nuovo dataset nella variabile df per usare sempre lui

# prendo una repository a caso e la analizzo
# random_repo = random.choice(repo_names)
# prendo una repo rappresentativa
random_repo = "joaomdmoura__crewai"

def print_repo_summary(repo_name: str) -> None:
    """
    Stampa una tabella per una singola repository con:
    - workflow_global_id
    - file_path (lista dei path osservati per quell'ID)
    - conteggio per tipo modifica (colonne: Aggiunto, Modificato, Eliminato, Rinominato)

    Requisiti:
    - usa CHANGE_TYPE_MAP per mappare i git_change_type
    - analizza solo la repository passata in input
    """
    repo_df = df[df["repository"] == repo_name].copy()

    if repo_df.empty:
        print(f"Nessun dato trovato per repository: {repo_name}")
        return

    # mapping human-readable
    repo_df["git_change_type_mapped"] = (
        repo_df["git_change_type"].map(CHANGE_TYPE_MAP).fillna(repo_df["git_change_type"])
    )

    # lista path per workflow_global_id
    paths_df = (
        repo_df.groupby("workflow_global_id", as_index=False)
        .agg(
            file_path=("file_path", lambda s: sorted(set(x for x in s.dropna() if str(x).strip())))
        )
    )

    # conteggi per tipo modifica per workflow_global_id
    counts_df = (
        repo_df.groupby(["workflow_global_id", "git_change_type_mapped"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    # ordine colonne preferito
    preferred_order = ["Aggiunto", "Modificato", "Eliminato", "Rinominato"]
    existing_preferred = [c for c in preferred_order if c in counts_df.columns]
    other_cols = [c for c in counts_df.columns if c not in (["workflow_global_id"] + existing_preferred)]

    counts_df = counts_df[["workflow_global_id"] + existing_preferred + other_cols]

    # merge finale
    summary = paths_df.merge(counts_df, on="workflow_global_id", how="left").fillna(0)

    print(f"Repository analizzata: {repo_name.split('__')[1] if '__' in repo_name else repo_name}")
    print(summary.to_string(index=False))

print_repo_summary(random_repo)

Repository analizzata: crewai
     workflow_global_id                                                   file_path  Aggiunto  Modificato  Eliminato  Rinominato
wf_1a596cccd50bb337b6be                            [.github/workflows/pr-title.yml]         1           0          0           0
wf_227bb8b9347d557ad5ef                             [.github/workflows/pr-size.yml]         1           0          0           0
wf_26d9370838f260778140                             [.github/workflows/publish.yml]         1           5          0           0
wf_2bfcf39e7dd8a3c21825                   [.github/workflows/notify-downstream.yml]         1           1          1           0
wf_2f1a238908e8496feab2                              [.github/workflows/codeql.yml]         1           3          0           0
wf_3a1180accd0886b879c4                  [.github/workflows/vulnerability-scan.yml]         1           1          0           0
wf_4e9da40c39fccdb2da3f                               [.github/work

In [12]:
# conto quante linee sono valid_workflow, quanti sono probably_workflow e quanti valid_yaml

valid_workflow_tr = df["valid_workflow"].where(df["valid_workflow"] == True).count()
print(f"Linee con valid_workflow = True: {valid_workflow_tr}, False = {len(df) - valid_workflow_tr}, Totale: {len(df)}")

probably_workflow = df["probably_workflow"].where(df["probably_workflow"] == True).count()
print(f"Linee con probably_workflow = True: {probably_workflow}, False = {len(df) - probably_workflow}, Totale: {len(df)}")

valid_yaml = df["valid_yaml"].where(df["valid_yaml"] == True).count()
print(f"Linee con valid_yaml = True: {valid_yaml}, False = {len(df) - valid_yaml}, Totale: {len(df)}")


# ogni singolo file con id univoco conto quante volte è stato valid_workflow, probably_workflow e valid_yaml
invalid_workflow_per_id = df["valid_workflow"].eq(False).groupby(df["workflow_global_id"]).sum()
probably_workflow_per_id = df["probably_workflow"].eq(True).groupby(df["workflow_global_id"]).sum()
valid_yaml_per_id = df["valid_yaml"].eq(True).groupby(df["workflow_global_id"]).sum()

print("media not_valid_workflow per workflow_global_id:", invalid_workflow_per_id.mean())
print("media probably_workflow per workflow_global_id:", probably_workflow_per_id.mean())
print("media valid_yaml per workflow_global_id:", valid_yaml_per_id.mean())

Linee con valid_workflow = True: 5817, False = 301, Totale: 6118
Linee con probably_workflow = True: 6115, False = 3, Totale: 6118
Linee con valid_yaml = True: 6080, False = 38, Totale: 6118
media not_valid_workflow per workflow_global_id: 0.4485842026825633
media probably_workflow per workflow_global_id: 9.113263785394933
media valid_yaml per workflow_global_id: 9.061102831594635


In [13]:
# valid_workflow è una dipendenza funzionale di valid_yaml?
# valid_yaml = True -> valid_workflow = True sempre ?

matrice_2x2 = (
    pd.crosstab(df["valid_workflow"], df["valid_yaml"], dropna=False)
    .reindex(index=[False, True], columns=[False, True], fill_value=0)
)

matrice_2x2.index.name = "valid_workflow"
matrice_2x2.columns.name = "valid_yaml"

print(matrice_2x2)

valid_yaml      False  True 
valid_workflow              
False              38    263
True                0   5817


#### df valid_yaml -> valid_workflow

possiamo dedurre che non sempre un file yaml valido indica un workflow valido, da tenere conto per l'analisi approfondita.

In [15]:
# funzione per calcolare il numero di file yaml in una repository
def count_yaml_files(repo_name):
    return len(repositories[repo_name])

In [ ]:
# funzione per calcolare il numero totale di jobs a partire dal file yaml
def count_jobs(yaml_file):
    pass

In [ ]:
# qui ci va funzione per calcolare il numero totale di jobs in una repository

In [ ]:
# funzione per calcolare il numero medio di jobs per workflow (file yaml)
def average_jobs_per_workflow(repo_name):
    pass

In [ ]:
# funzione per ottenere il numero di eventi CONSIDERATI in una repository
# ad esempio, se un workflow è triggerato da push o pull

def count_unique_events(repo_name):
    '''
    questa funzione dovrebbe restituire il NUMERO di eventi unici, ma forse è meglio restituire la lista degli eventi
    '''
    pass


In [ ]:
# funzione per ottenere le runner labels, quindi i vari sistemi in cui vengono eseguiti i workflow

def get_runner_labels(repo_name):
    pass

In [ ]:
# funzione per capire se il workflow usa matrix strategy

# ricordando che per matrix si intende: La strategia di matrice permette di eseguire lo stesso job più volte contemporaneamente, utilizzando diverse combinazioni di variabili (come versioni di linguaggi, sistemi operativi o librerie).

def uses_matrix_strategy_file(yaml_file):
    pass

def uses_matrix_strategy(repo_name):
    ''' fondamentalmente questa funzione va a iterare su tutti i file nella repo chiamando la funzione sopra'''
    pass

In [ ]:
# funzione per capire se c'è workflow dispatch abilitato

# quindi se è possibile avviare manualmente un workflow (a prescindere dai trigger)

def has_workflow_dispatch_file(yaml_file):
    pass

def has_workflow_dispatch(repo_name):
    ''' fondamentalmente questa funzione va a iterare su tutti i file nella repo chiamando la funzione sopra'''
    pass


In [ ]:
# funzione per capire se sono presenti automazioni che si eseguono in certi orari (tipo cronjob)

def has_workflow_dispatch_file(yaml_file):
    pass

def has_workflow_dispatch(repo_name):
    ''' fondamentalmente questa funzione va a iterare su tutti i file nella repo chiamando la funzione sopra'''
    pass


In [ ]:
#funzione per capire se c'è abilitato workflow call

#Indica che il repository offre i suoi workflow ad altri. Funziona come una "funzione" pubblica: un altro file può chiamare questo workflow passandogli dei dati.

def has_workflow_call_file(yaml_file):
    pass

def has_workflow_call(repo_name):
    ''' fondamentalmente questa funzione va a iterare su tutti i file nella repo chiamando la funzione sopra'''
    pass

In [ ]:
# funzione per capire se c'è workflow riusabile

#Indica che il repository chiama workflow esterni o interni per non duplicare il codice. Invece di scrivere 100 righe di passaggi, ne usa una sola che richiama un "modello" standard già pronto.

def has_reusable_workflow_file(yaml_file):
    pass

def has_reusable_workflow(repo_name):
    ''' fondamentalmente questa funzione va a iterare su tutti i file nella repo chiamando la funzione sopra'''
    pass

In [ ]:
# funzione per unique_actions_count
# È il numero di strumenti diversi (Action) prelevati dal Marketplace o dal repository stesso (es. actions/checkout, docker/build-push-action). Più è alto, più il workflow è "composto" da moduli specializzati.

def unique_actions_count(repo_name):
    pass

# Sezione metriche vere e proprie

da approfondire la questione.

In [ ]:
# Indica quanti blocchi condizionali (if:) ci sono in media per ogni file. Un valore alto (come 4.9) suggerisce workflow molto dinamici che eseguono passaggi diversi a seconda della situazione (es. "se è un fallimento fai questo", "se è il branch main fai quello").

def average_conditionals_per_workflow(repo_name):
    pass

In [ ]:
# Indica quante dipendenze tra job ci sono. Se un job ha bisogno (needs) che un altro finisca prima di partire, la pipeline è sequenziale o a grafo. Più è alto, più i job sono interconnessi e non indipendenti.
def needs_per_workflow(repo_name):
    pass


In [ ]:
# uses per workflow
# Conta quante volte vengono richiamate delle Action (sia esterne che locali). Più è alto più il worflow è frammentato in pezzi

def uses_per_workflow(repo_name):
    pass

In [ ]:
# external action ratio
# Indica la percentuale di action che vengono dal marketplace o da repository esterne. quindi più è alto più il workflow si affida a soluzioni esterne invece di implementare tutto in casa.

def external_action_ratio(repo_name):
    pass

In [ ]:
# local action ratio
# Indica la percentuale di action che vengono implementate all'interno del repository. Quindi più è alto più il workflow si affida a soluzioni interne invece di prendere tutto dal marketplace-

def local_action_ratio(repo_name):
    pass

In [ ]:
# unique external actions count
# numero di action uniche che vengono dall'esterno, perchè il ratio potrebbe essere alto anche se poi magari prendo 100 volte la stessa action

def unique_external_actions_count(repo_name):
    pass

In [ ]:
# complexity score
# indice di complessità, formula:
"""
complexity = (
            0.30 * n_avg_jobs[repo]
            + 0.15 * n_events[repo]
            + 0.10 * n_runners[repo]
            + 0.20 * n_if_per_wf[repo]
            + 0.15 * n_needs_per_wf[repo]
            + 0.10 * has_matrix
        )
"""

def complexity_score(repo_name):
    pass

In [ ]:
# glue code score
# un altro indice, che misura quanto il workflow si affida a pezzi esterni (Action) invece di implementare tutto in casa, formula:

"""
glue = (
            0.20 * n_actions[repo]
            + 0.20 * n_uses_per_wf[repo]
            + 0.20 * ext_ratio[repo]
            + 0.10 * local_ratio[repo]
            + 0.15 * n_uniq_external[repo]
            + 0.10 * uses_reusable
            + 0.05 * has_workflow_call
        )
"""

In [ ]:
# mantainability risk score
# indice che combina complexity e glue

"""
risk = (
            0.45 * complexity
            + 0.35 * glue
            + 0.20 * n_wf[repo]
        )
"""